In [1]:
import polars as pl
import datetime as dt
import random

from typing import Literal

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
now = dt.datetime(2026, 1, 1)
expiry = dt.datetime(2026, 1, 2)
symbol = 'BTC'

### Value block data structure

In [3]:
def annualize(value, seconds):
    return value * (365.25 * 24 * 60 * 60) / seconds

def deannualize(value, seconds):
    return value / (365.25 * 24 * 60 * 60) * seconds

class ValueBlock():
    def __init__(
        #-----Static parameters-----#
        # defaults are for simple base vol pricing
        self,
        block_name:str,
        annualized:bool=True,
        size_type:Literal['fixed', 'relative']='fixed', # Fixed size or size relative to market implied
        aggregation_logic:Literal['average', 'offset']='average', # i.e., fundamental/realised (average all fundamental valuations) or implied (sum all expected price moves)
        temporal_position:Literal['static', 'shifting']='shifting', # if shifting, start timestamp is always now
        decay_end_size_mult:float=1.0, # can only be non-zero for annualized; decay start size is calculated by size, since total variance contribution is fixed
        decay_rate_prop_per_min:float=0.0, # speed of total variance decay
        decay_profile:Literal['linear']='linear', # decay is in total variance space, so actually looks like uniform block on variance x timestamp graph
        start_timestamp:dt.datetime=None, # only applicable for static streams
        
        #----- Dynamic parameters -----#
        var_fair_ratio:float=1.0,
    ):
        self.block_name = block_name
        self.annualized = annualized
        self.size_type = size_type
        self.aggregation_logic = aggregation_logic
        self.temporal_position = temporal_position
        self.decay_end_size_mult = decay_end_size_mult
        self.decay_rate_prop_per_min = decay_rate_prop_per_min
        self.decay_profile = decay_profile
        self.start_timestamp = start_timestamp
        
        self.var_fair_ratio = var_fair_ratio
        
        self._check_attributes()
        
    def _check_attributes(self):
        # Choice variables
        assert self.size_type in ['fixed', 'relative'], "size_type must be 'fixed' or 'relative'"
        assert self.aggregation_logic in ['average', 'offset'], "aggregation_logic must be 'average' or 'offset'"
        assert self.temporal_position in ['static', 'shifting'], "temporal_position must be 'static' or 'shifting'"
        assert self.decay_profile in ['linear'], "decay_profile must be 'linear'"
        
        # Numerical variables
        assert self.decay_end_size_mult >= 0, "decay_end_size_mult must be non-negative"
        assert self.decay_rate_prop_per_min >= 0, "decay_rate_prop_per_min must be non-negative"
        
        # Special conditions
        if self.size_type == 'relative': # not necessary, can adapt for total type streams
            assert self.annualized, " must be annualized for relative size"
        if self.decay_end_size_mult != 0 and not self.annualized:
            raise ValueError("decay_end_size_mult is only applicable for annualized streams")
        if self.temporal_position == "static" and self.start_timestamp is None:
            raise ValueError("start_timestamp must be provided for static streams")
        
        
    def _get_end_timestamp(self, start_ts:dt.datetime, expiry:dt.datetime):
        if self.decay_end_size_mult == 1 or self.decay_rate_prop_per_min == 0:
            return expiry
        else:
            return start_ts + dt.timedelta(minutes=1 / self.decay_rate_prop_per_min)
        
        
    def _get_total_value(self, stream_value:float, market_value:float, start_ts:dt.datetime, end_ts:dt.datetime):
        if self.annualized: # Relative pricing not applicable for total type stream, since we're assuming market_value is always annualised
            annualized_value = stream_value if self.size_type=='fixed' else stream_value - market_value
            return deannualize(annualized_value, (end_ts - start_ts).total_seconds())
        else:
            return stream_value
        
        
    def _get_start_annualized_value(self, total_value:float, expiry:dt.datetime, start_ts:dt.datetime, end_ts:dt.datetime, end_annualized_value:float):
        start_to_expiry_secs = (expiry - start_ts).total_seconds()
        start_to_end_secs = (end_ts - start_ts).total_seconds()
        annualized_value = annualize(total_value, start_to_end_secs)
            
        if self.annualized:
            start_to_end_secs = min(start_to_expiry_secs, start_to_end_secs)
            # Stream Value = Prop. time between start and end * (Average of Start and End Value, assuming linear) + (1-Prop.) * End Value
            # v = p * (s + e) / 2 + (1 - p) * e
            # ==> s = (2 / p) * (v - (1 - p) * e) - e
            p = start_to_end_secs / start_to_expiry_secs
            return (2 / p) * (annualized_value - (1 - p) * end_annualized_value) - end_annualized_value
        else:
            return annualized_value
        
    
    def calculate_fair(self, now:dt.datetime, stream_value:float, market_value:float, future_df:pl.DataFrame, expiry:dt.datetime):
        # Determine timestamp range
        start_ts = now if self.temporal_position == "shifting" else self.start_timestamp
        end_ts = self._get_end_timestamp(start_ts, expiry)
        
        # Calculate total value (e.g., total variance)
        total_value = self._get_total_value(stream_value, market_value, start_ts, end_ts)
        
        # Calculate start and end instantaneous annualised variance
        end_annualized_value = annualize(total_value, (end_ts - start_ts).total_seconds()) * self.decay_end_size_mult # doesn't matter that stream_value isn't annualised for 'total' type streams, since mult will always be 0
        start_annualized_value = self._get_start_annualized_value(total_value, expiry, start_ts, end_ts, end_annualized_value)
        
        # Add column to future_df for fair value
        out_df = future_df.with_columns(
            (
                pl.when(pl.col('timestamp') < start_ts)
                .then(start_annualized_value)
                .when(pl.col('timestamp') > end_ts)
                .then(end_annualized_value)
                .when(self.annualized)
                .then(
                    # only applicable for linear
                    start_annualized_value + (end_annualized_value - start_annualized_value) * (pl.col('timestamp') - start_ts) / (end_ts - start_ts)
                )
                .otherwise(start_annualized_value) # only applicable for linear
            ).alias(f'{self.block_name}_fair_annualized')
        ).with_columns(
            (pl.col(f'{self.block_name}_fair_annualized') * pl.col('dtte')).alias(f'{self.block_name}_fair')
        )
        return out_df
    
    
    def calculate_variance(self, future_df:pl.DataFrame):
        return future_df.with_columns(
            (pl.col(f'{self.block_name}_fair') * self.var_fair_ratio).alias(f'{self.block_name}_var')
        )
    
    
    def calculate_fair_and_variance(self, now:dt.datetime, stream_value:float, market_value:float, future_df:pl.DataFrame, expiry:dt.datetime):
        out_df = self.calculate_fair(now, stream_value, market_value, future_df, expiry)
        out_df = self.calculate_variance(out_df)
        return out_df

### Data Stream data structure

In [4]:
def general_raw_to_target_transformation(raw_value:float, scale:float, offset:float, exponent:float) -> float:
    return (scale * raw_value + offset)**exponent

class DataStream():
    def __init__(
            self,
            stream_name:str,
            snapshot:pl.DataFrame, 
            key_cols:list[str],
            scale:float=None,
            offset:float=None,
            exponent:float=None,
            annualized:bool=None,
            
            # Value blocks
            size_type:Literal['fixed', 'relative']='fixed', # Fixed size or size relative to market implied
            aggregation_logic:Literal['average', 'offset']='average', # i.e., fundamental/realised (average all fundamental valuations) or implied (sum all expected price moves)
            temporal_position:Literal['static', 'shifting']='shifting', # if shifting, start timestamp is always now
            decay_end_size_mult:float=1.0, # can only be non-zero for annualized; decay start size is calculated by size, since total variance contribution is fixed
            decay_rate_prop_per_min:float=0.0, # speed of total variance decay
            decay_profile:Literal['linear']='linear', # decay is in total variance space, so actually looks like uniform block on variance x timestamp graph
            start_timestamp:dt.datetime=None, # only applicable for static streams
            var_fair_ratio:float=1.0,
        ):
        self.stream_name = stream_name
        self.snapshot = snapshot
        self.key_cols = key_cols
        self.extra_key_cols = [k for k in key_cols if k not in ('symbol', 'expiry')]
        
        # To be determined by LLM
        self.scale  = scale
        self.offset = offset
        self.exponent = exponent
        self.annualized = annualized
        
        # Value blocks parameters
        self.size_type = size_type
        self.aggregation_logic = aggregation_logic
        self.temporal_position = temporal_position
        self.decay_end_size_mult = decay_end_size_mult
        self.decay_rate_prop_per_min = decay_rate_prop_per_min
        self.decay_profile = decay_profile
        self.start_timestamp = start_timestamp
        self.var_fair_ratio = var_fair_ratio
        
        self.update_snapshot(self.snapshot)
        
    def update_snapshot(self, snapshot:pl.DataFrame):
        self.snapshot = snapshot
        self.get_latest_snapshot()
        self.transform_raw_to_target()
        self.get_value_blocks()
        
    def get_latest_snapshot(self):
        self.snapshot = self.snapshot.sort('timestamp').group_by(self.key_cols).agg(pl.all().last())
        
    def transform_raw_to_target(self):
        self.snapshot = self.snapshot.with_columns(
            pl.col('raw_value').map_elements(lambda x: general_raw_to_target_transformation(x, self.scale, self.offset, self.exponent)).alias('target_value')
        )
        
    def _make_block_name(self, row:dict) -> str:
        parts = [self.stream_name]
        for k in self.extra_key_cols:
            parts.append(str(row[k]))
        return "_".join(parts)
        
    def get_value_blocks(self):
        block_map = {}
        for keys, group in self.snapshot.group_by(self.key_cols):
            key_tuple = keys if isinstance(keys, tuple) else (keys,)
            row = dict(zip(self.key_cols, key_tuple))
            block_name = self._make_block_name(row)
            block_map[key_tuple] = ValueBlock(
                block_name=block_name,
                annualized=self.annualized,
                size_type=self.size_type,
                aggregation_logic=self.aggregation_logic,
                temporal_position=self.temporal_position,
                decay_end_size_mult=self.decay_end_size_mult,
                decay_rate_prop_per_min=self.decay_rate_prop_per_min,
                decay_profile=self.decay_profile,
                start_timestamp=self.start_timestamp,
                var_fair_ratio=self.var_fair_ratio,
            )
    
        self.snapshot = self.snapshot.with_columns(
            pl.struct(self.key_cols)
            .map_elements(lambda row: block_map[tuple(row.values())], return_dtype=pl.Object)
            .alias("value_block")
        )

### Stream initialisation + Conversion to target space

In [5]:
# Snapshots are sent per data stream

mock_timestamp = now

RVStream = DataStream(
    'rv',
    pl.DataFrame({
        'timestamp' : mock_timestamp,
        'symbol' : symbol,
        'expiry' : expiry,
        'raw_value' : random.random(),
    }),
    ['symbol', 'expiry'],
    scale=1, offset=0, exponent=2,
    annualized=True
)
MeanIVStream = DataStream(
    'mean_iv',
    pl.DataFrame({
        'timestamp' : mock_timestamp,
        'symbol' : symbol,
        'expiry' : expiry,
        'raw_value' : random.random(),
    }),
    ['symbol', 'expiry'],
    scale=1, offset=0, exponent=2,
    annualized=True, aggregation_logic='offset', size_type='relative',
    decay_end_size_mult=0.0, decay_rate_prop_per_min=0.01, decay_profile='linear',
    var_fair_ratio=2.0 # assume we're less confidence in this -- less confident = higher variance
)

num_events = 5
EventsStream = DataStream(
    'events',
    pl.DataFrame({
        'timestamp' : [mock_timestamp for _ in range(num_events)],
        'symbol' : symbol,
        'expiry' : expiry,
        'event_id' : [f'event_{i}' for i in range(num_events)],
        'raw_value' : [random.random() * 5 for _ in range(num_events)],
    }),
    ['symbol', 'expiry', 'event_id'],
    scale=1/100, offset=0, exponent=2,
    annualized=False, aggregation_logic='offset', size_type='fixed',
    decay_end_size_mult=0.0, decay_rate_prop_per_min=0.01, decay_profile='linear',
    var_fair_ratio=3.0
)

data_streams = [RVStream, MeanIVStream, EventsStream]

for stream in data_streams:
    display(stream.snapshot)

symbol,expiry,timestamp,raw_value,target_value,value_block
str,datetime[μs],datetime[μs],f64,f64,object
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.318022,0.101138,<__main__.ValueBlock object at 0x1071c2660>


symbol,expiry,timestamp,raw_value,target_value,value_block
str,datetime[μs],datetime[μs],f64,f64,object
"""BTC""",2026-01-02 00:00:00,2026-01-01 00:00:00,0.934438,0.873174,<__main__.ValueBlock object at 0x106449310>


symbol,expiry,event_id,timestamp,raw_value,target_value,value_block
str,datetime[μs],str,datetime[μs],f64,f64,object
"""BTC""",2026-01-02 00:00:00,"""event_2""",2026-01-01 00:00:00,4.989307,0.002489,<__main__.ValueBlock object at 0x1074c97b0>
"""BTC""",2026-01-02 00:00:00,"""event_4""",2026-01-01 00:00:00,2.402984,0.000577,<__main__.ValueBlock object at 0x1074c0b00>
"""BTC""",2026-01-02 00:00:00,"""event_3""",2026-01-01 00:00:00,3.812519,0.001454,<__main__.ValueBlock object at 0x1074d1250>
"""BTC""",2026-01-02 00:00:00,"""event_1""",2026-01-01 00:00:00,2.289082,0.000524,<__main__.ValueBlock object at 0x106449590>
"""BTC""",2026-01-02 00:00:00,"""event_0""",2026-01-01 00:00:00,0.837473,0.00007,<__main__.ValueBlock object at 0x1074c0fc0>


### Fair value / variance conversion

In [6]:
future_df = pl.DataFrame({
    'timestamp' : pl.datetime_range(
        start=now,
        end=expiry,
        interval='1s',
        eager=True
    ),
    'symbol' : symbol,
    'expiry' : expiry,
}).with_columns(
    dtte=-pl.col('timestamp').diff(-1).dt.total_seconds() / (60 * 60 * 24 * 365.25),
    current_market_implied_vol=pl.lit(0.5)
)
future_df

timestamp,symbol,expiry,dtte,current_market_implied_vol
datetime[μs],str,datetime[μs],f64,f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
…,…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5


In [7]:
# Enrich future_df with fair/var from each DataStream's value blocks, then aggregate

enriched_parts = []
for (sym, exp), sub_df in future_df.group_by('symbol', 'expiry'):
    mkt_var = sub_df['current_market_implied_vol'][0] ** 2
    applied_blocks = []
    
    for stream in data_streams:
        matches = stream.snapshot.filter(
            (pl.col('symbol') == sym) & (pl.col('expiry') == exp)
        )
        for row in matches.iter_rows(named=True):
            block = row['value_block']
            sub_df = block.calculate_fair_and_variance(
                now=sub_df['timestamp'].min(),
                stream_value=row['target_value'],
                market_value=mkt_var,
                future_df=sub_df,
                expiry=exp
            )
            applied_blocks.append(block)
    
    # Aggregate: average vs offset fair, cumulative variance
    sub_df = sub_df.with_columns(
        mkt_implied_annualized=pl.lit(mkt_var),
        mkt_implied=pl.lit(mkt_var) * pl.col('dtte'),
        average_fair=pl.lit(0.0),
        average_fair_count=pl.lit(0),
        offset_fair=pl.lit(0.0),
        cum_var=pl.lit(0.0),
    )
    for block in applied_blocks:
        if block.aggregation_logic == 'average':
            sub_df = sub_df.with_columns(
                average_fair=pl.col('average_fair') + pl.col(f'{block.block_name}_fair'),
                average_fair_count=pl.col('average_fair_count') + 1,
            )
        elif block.aggregation_logic == 'offset':
            sub_df = sub_df.with_columns(
                offset_fair=pl.col('offset_fair') + pl.col(f'{block.block_name}_fair'),
            )
        sub_df = sub_df.with_columns(
            cum_var=pl.col('cum_var') + pl.col(f'{block.block_name}_var'),
        )
    
    sub_df = sub_df.with_columns(
        edge=pl.when(pl.col('average_fair_count') > 0)
            .then(pl.col('offset_fair') + pl.col('average_fair') / pl.col('average_fair_count') - pl.col('mkt_implied'))
            .otherwise(pl.col('offset_fair')),
        var=pl.col('cum_var'),
    )
    enriched_parts.append(sub_df)

fair_var_df = pl.concat(enriched_parts)
display(fair_var_df)

timestamp,symbol,expiry,dtte,current_market_implied_vol,rv_fair_annualized,rv_fair,rv_var,mean_iv_fair_annualized,mean_iv_fair,mean_iv_var,events_event_2_fair_annualized,events_event_2_fair,events_event_2_var,events_event_4_fair_annualized,events_event_4_fair,events_event_4_var,events_event_3_fair_annualized,events_event_3_fair,events_event_3_var,events_event_1_fair_annualized,events_event_1_fair,events_event_1_var,events_event_0_fair_annualized,events_event_0_fair,events_event_0_var,mkt_implied_annualized,mkt_implied,average_fair,average_fair_count,offset_fair,cum_var,edge,var
datetime[μs],str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.94741,5.6872e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.944418,5.6862e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.941427,5.6853e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.938436,5.6843e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.935445,5.6834e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9


### Desired position calculation

In [8]:
smoothing_hl_secs = 60 * 15
bankroll = 100_000

In [9]:
desired_position_df = fair_var_df.with_columns(
    raw_desired_position = pl.col('edge') * bankroll / pl.col('var')
).with_columns(
    smoothed_desired_position=pl.col('raw_desired_position').reverse().ewm_mean_by('timestamp', half_life=f'{smoothing_hl_secs}s').reverse()
)
desired_position_df

timestamp,symbol,expiry,dtte,current_market_implied_vol,rv_fair_annualized,rv_fair,rv_var,mean_iv_fair_annualized,mean_iv_fair,mean_iv_var,events_event_2_fair_annualized,events_event_2_fair,events_event_2_var,events_event_4_fair_annualized,events_event_4_fair,events_event_4_var,events_event_3_fair_annualized,events_event_3_fair,events_event_3_var,events_event_1_fair_annualized,events_event_1_fair,events_event_1_var,events_event_0_fair_annualized,events_event_0_fair,events_event_0_var,mkt_implied_annualized,mkt_implied,average_fair,average_fair_count,offset_fair,cum_var,edge,var,raw_desired_position,smoothed_desired_position
datetime[μs],str,datetime[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i32,f64,f64,f64,f64,f64,f64
2026-01-01 00:00:00,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.94741,5.6872e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004,38303.458341,35639.337681
2026-01-01 00:00:01,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.944418,5.6862e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004,38302.858678,35637.285082
2026-01-01 00:00:02,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.941427,5.6853e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004,38302.258953,35635.231364
2026-01-01 00:00:03,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.938436,5.6843e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004,38301.659167,35633.176525
2026-01-01 00:00:04,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,17.935445,5.6834e-7,0.000001,13.092818,4.1489e-7,0.000001,3.037067,9.6239e-8,2.8872e-7,7.644986,2.4225e-7,7.2676e-7,2.755975,8.7332e-8,2.6199e-7,0.368887,1.1689e-8,3.5068e-8,0.25,7.9220e-9,3.2049e-9,1,0.000001,0.000004,0.000001,0.000004,38301.059319,35631.120565
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-01-01 23:59:56,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9,-147187.320919,-147187.320919
2026-01-01 23:59:57,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9,-147187.320919,-147187.320919
2026-01-01 23:59:58,"""BTC""",2026-01-02 00:00:00,3.1688e-8,0.5,0.101138,3.2049e-9,3.2049e-9,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.25,7.9220e-9,3.2049e-9,1,0.0,3.2049e-9,-4.7172e-9,3.2049e-9,-147187.320919,-147187.320919


# Sensechecking

In [10]:
### Sensechecking: Fair Value, Variance, Edge

# Collect all ValueBlock objects from DataStreams
blocks = []
for stream in data_streams:
    for row in stream.snapshot.iter_rows(named=True):
        blocks.append(row['value_block'])

# Downsample to minutely for plotting (86K → ~1440 points)
plot_df = fair_var_df.gather_every(60).drop_nulls(subset=['dtte'])
ts = plot_df['timestamp'].to_list()

In [11]:
# Graph 1: Fair Value by Stream vs Market Implied
fig1 = go.Figure()

avg_blocks = [b for b in blocks if b.aggregation_logic == 'average']
offset_blocks = [b for b in blocks if b.aggregation_logic == 'offset']

# Market implied reference
fig1.add_trace(go.Scatter(
    x=ts, y=plot_df['mkt_implied'].to_list(),
    name='mkt implied',
    line=dict(dash='dash', color='rgba(255,255,255,0.6)', width=2)
))

# Average blocks — overlapping
for b in avg_blocks:
    fig1.add_trace(go.Scatter(
        x=ts, y=plot_df[f'{b.block_name}_fair'].to_list(),
        name=f'{b.block_name} fair', mode='lines'
    ))

# Combined base (avg of average blocks)
base = plot_df['average_fair'] / plot_df['average_fair_count']

# Base trace (anchor for offset fill)
fig1.add_trace(go.Scatter(
    x=ts, y=base.to_list(),
    name='base avg', line=dict(width=1, dash='dot')
))

# Offset blocks — stacked on base
cumulative = base.clone()
for b in offset_blocks:
    cumulative = cumulative + plot_df[f'{b.block_name}_fair']
    fig1.add_trace(go.Scatter(
        x=ts, y=cumulative.to_list(),
        name=f'+ {b.block_name} (offset)',
        fill='tonexty'
    ))

fig1.update_layout(
    title='Fair Value by Stream vs Market Implied',
    template='plotly_dark',
    xaxis_title='Time', yaxis_title='Variance (per interval)'
)
fig1.show()

In [12]:
# Graph 2: Stacked Variance Contribution by Stream
fig2 = go.Figure()

for b in blocks:
    fig2.add_trace(go.Scatter(
        x=ts, y=plot_df[f'{b.block_name}_var'].to_list(),
        name=f'{b.block_name} var',
        stackgroup='one'
    ))

fig2.update_layout(
    title='Variance Contribution by Stream',
    template='plotly_dark',
    xaxis_title='Time', yaxis_title='Variance (per interval)'
)
fig2.show()

In [13]:
# Graph 3: Edge & Variance
fig3 = make_subplots(specs=[[{"secondary_y": True}]])

fig3.add_trace(
    go.Scatter(x=ts, y=plot_df['edge'].to_list(), name='edge'),
    secondary_y=False
)
fig3.add_trace(
    go.Scatter(x=ts, y=plot_df['var'].to_list(), name='var'),
    secondary_y=True
)

fig3.update_layout(title='Edge & Variance', template='plotly_dark')
fig3.update_xaxes(title_text='Time')
fig3.update_yaxes(title_text='Edge', secondary_y=False)
fig3.update_yaxes(title_text='Variance', secondary_y=True)
fig3.show()

In [14]:
# Graph 4: Raw and Smoothed Desired Position
pos_plot_df = desired_position_df.gather_every(60).drop_nulls(subset=['dtte'])
pos_ts = pos_plot_df['timestamp'].to_list()

fig4 = go.Figure()

fig4.add_trace(
    go.Scatter(x=pos_ts, y=pos_plot_df['raw_desired_position'].to_list(),
               name='Raw Desired Position', line=dict(width=1, color='rgba(99,110,250,0.4)'))
)
fig4.add_trace(
    go.Scatter(x=pos_ts, y=pos_plot_df['smoothed_desired_position'].to_list(),
               name='Smoothed Desired Position', line=dict(width=2))
)

fig4.update_layout(
    title='Raw and Smoothed Desired Position',
    template='plotly_dark',
    xaxis_title='Time', yaxis_title='Desired Position ($)'
)
fig4.show()